In [1]:
import tensorflow as tf
import numpy as np

In [2]:
model = tf.keras.models.load_model(
    '/content/drive/MyDrive/Colab Notebooks/CS AVANZADAS/Modulo Inteligencia Artificial/Proyecto/bnpl_model_v2.keras'
)

model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                │ (None, 64)             │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,685 (30.02 KB)

 Trainable params: 2,561 (10.00 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 5,124 (20.02 KB)

In [3]:
# Cargar escalamiento
mean = np.load('/content/drive/MyDrive/Colab Notebooks/CS AVANZADAS/Modulo Inteligencia Artificial/Proyecto/mean.npy')
std  = np.load('/content/drive/MyDrive/Colab Notebooks/CS AVANZADAS/Modulo Inteligencia Artificial/Proyecto/std.npy')

In [14]:
print("Predictor de Riesgo BNPL \n")

while True:
    risk_score = float(input("¿Cual es el score de riesgo (Risk score) del cliente? (0–398): "))
    if 0 <= risk_score <= 398:
        break
    else:
        print("Por favor, ingrese un score de riesgo entre 0 y 398.")

while True:
    credit_score = float(input("¿Cual es el puntaje crediticio del cliente? (300–850): "))
    if 300 <= credit_score <= 850:
        break
    else:
        print("Por favor, ingrese un puntaje crediticio entre 300 y 850.")

while True:
    repayment = float(input("¿Cuantos días de retraso en pagos anteriores ha tenido el cliente? (0–33): "))
    if 0 <= repayment <= 33:
        break
    else:
        print("Por favor, ingrese un número de días de retraso entre 0 y 33.")

while True:
    income = float(input("¿Cual es el ingreso mensual del cliente? (USD):$ "))
    if income > 0:
        break
    else:
        print("Por favor, ingrese un ingreso mensual mayor que 0.")

while True:
    missed = float(input("¿Cuantos pagos ha omitido el cliente? (0–7): "))
    if 0 <= missed <= 7:
        break
    else:
        print("Por favor, ingrese un número de pagos omitidos entre 0 y 7.")

while True:
    debt = float(input("¿Cual es la deuda total mensual del cliente? (USD):$ "))
    if debt >= 0:
        dti = np.clip(debt / income, 0.0, 1.0)
        break
    else:
        print("Por favor, ingrese una deuda mensual no negativa.")

usuario = np.array([[risk_score, credit_score, repayment,
                     income, missed, dti]], dtype=np.float32)

# Esto es para tener el mismo escalamiento a los parametros del entrenamiento (mean y std)
usuario_scaled = (usuario - mean) / std


Predictor de Riesgo BNPL 

¿Cual es el score de riesgo (Risk score) del cliente? (0–398): 200
¿Cual es el puntaje crediticio del cliente? (300–850): 550
¿Cuantos días de retraso en pagos anteriores ha tenido el cliente? (0–33): 10
¿Cual es el ingreso mensual del cliente? (USD):$ 20000
¿Cuantos pagos ha omitido el cliente? (0–7): 3
¿Cual es la deuda total mensual del cliente? (USD):$ 10000


In [15]:
output = model(usuario_scaled, training=False).numpy()
output

array([[0.9653363]], dtype=float32)

In [16]:
primer_usuario = output[0]
primer_usuario

array([0.9653363], dtype=float32)

In [17]:
probabilidad = primer_usuario[0]
probabilidad

np.float32(0.9653363)

In [18]:
print(f"Probabilidad de default: {probabilidad:.1%}")
print(f"Prediccion: {'Default' if probabilidad >= 0.5 else 'Pagará'}")

Probabilidad de default: 96.5%
Prediccion: Default
